# LangChain y LangSmith · Trazas y evaluación con LangSmith

[LangSmith](https://docs.langchain.com/langsmith/home) es la plataforma de LangChain para **observar** aplicaciones con modelos de lenguaje (trazas de cada paso con sus entradas, salidas, tiempos y errores) y **evaluarlas** con conjuntos de ejemplos y evaluadores. El SDK `langsmith` funciona con cualquier código Python, use LangChain o no.

Normalmente las trazas se envían a LangSmith definiendo `LANGSMITH_TRACING=true` y `LANGSMITH_API_KEY`. Aquí todo se ejecuta en local y sin clave: las trazas se construyen con `tracing_context(enabled="local")` y se inspeccionan en Python, y las evaluaciones usan `evaluate(..., upload_results=False)`. Con una clave, el mismo código las sube a la plataforma.

## 1. Trazar una función

Decorar una función con `@traceable` y ejecutarla dentro de `tracing_context(enabled="local")`, recogiendo la traza terminada con `langsmith_extra={"on_end": ...}`. Mostrar el nombre, el tipo, las entradas, las salidas y la duración de la ejecución, y comprobar que fuera del contexto la función se comporta igual pero no deja traza.

In [1]:
import warnings

import langsmith as ls
from langsmith import traceable
from langsmith.utils import LangSmithMissingAPIKeyWarning

warnings.filterwarnings("ignore", category=LangSmithMissingAPIKeyWarning)
warnings.filterwarnings("ignore", message=".*'upload_results' parameter is in beta")


@traceable(run_type="tool", name="Contar vocales")
def contar_vocales(texto: str) -> dict:
    return {v: texto.lower().count(v) for v in "aeiou" if v in texto.lower()}


trazas = []
with ls.tracing_context(enabled="local"):
    resultado = contar_vocales("Roque Nublo", langsmith_extra={"on_end": trazas.append})

traza = trazas[0]
print(resultado)
print(f"{traza.name} [{traza.run_type}] entradas={traza.inputs} salidas={traza.outputs}")
print(f"duración: {(traza.end_time - traza.start_time).total_seconds() * 1000:.3f} ms, id={traza.id}")

contar_vocales("Garajonay", langsmith_extra={"on_end": trazas.append})
assert len(trazas) == 1

{'e': 1, 'o': 2, 'u': 2}
Contar vocales [tool] entradas={'texto': 'Roque Nublo'} salidas={'e': 1, 'o': 2, 'u': 2}
duración: 1.688 ms, id=01a1d13d-ebde-75e3-aaaf-d9dfc4aebd6e


## 2. Árbol de trazas

Las funciones trazadas que se llaman entre sí forman un árbol. Escribir un pequeño asistente en Python puro con tres pasos trazados (`normalizar`, `recuperar` de tipo `retriever` y `generar` de tipo `llm`) dentro de una función `responder` de tipo `chain`. Escribir `mostrar_arbol(traza)`, que imprima el árbol con sangría, el tipo y la duración de cada paso, y comprobar el orden de los hijos.

In [2]:
import re
import time
import unicodedata

HECHOS = [
    "La capital de Lanzarote es Arrecife.",
    "La capital de Fuerteventura es Puerto del Rosario.",
    "La capital de La Gomera es San Sebastián de La Gomera.",
    "La capital de El Hierro es Valverde.",
    "El Teide mide 3715 metros.",
    "El Roque Nublo está en Gran Canaria.",
    "La Graciosa es la isla habitada más pequeña.",
]


def sin_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")


@traceable
def normalizar(pregunta: str) -> set[str]:
    return {p for p in re.findall(r"\w+", sin_tildes(pregunta.lower())) if len(p) > 3}


@traceable(run_type="retriever")
def recuperar(palabras: set[str], k: int = 2) -> list[dict]:
    puntuados = sorted(HECHOS, key=lambda h: -len(palabras & normalizar.__wrapped__(h)))
    return [{"page_content": h, "type": "Document", "metadata": {}} for h in puntuados[:k]]


@traceable(run_type="llm")
def generar(pregunta: str, documentos: list[dict]) -> str:
    time.sleep(0.01)
    return documentos[0]["page_content"]


@traceable(run_type="chain")
def responder(pregunta: str) -> str:
    return generar(pregunta, recuperar(normalizar(pregunta)))


def mostrar_arbol(traza, nivel=0):
    ms = (traza.end_time - traza.start_time).total_seconds() * 1000
    print(f"{'    ' * nivel}{traza.name} [{traza.run_type}] {ms:.1f} ms")
    for hijo in traza.child_runs:
        mostrar_arbol(hijo, nivel + 1)


trazas = []
with ls.tracing_context(enabled="local"):
    print(responder("¿Cuál es la capital de El Hierro?", langsmith_extra={"on_end": trazas.append}))
mostrar_arbol(trazas[0])
assert [h.name for h in trazas[0].child_runs] == ["normalizar", "recuperar", "generar"]
print(trazas[0].child_runs[1].outputs)

La capital de El Hierro es Valverde.
responder [chain] 11.2 ms
    normalizar [chain] 0.1 ms
    recuperar [retriever] 0.1 ms
    generar [llm] 10.7 ms
{'output': [{'page_content': 'La capital de El Hierro es Valverde.', 'type': 'Document', 'metadata': {}}, {'page_content': 'La capital de Lanzarote es Arrecife.', 'type': 'Document', 'metadata': {}}]}


## 3. Etiquetas y metadatos

Añadir etiquetas y metadatos a las trazas de cuatro formas: fijos en el decorador, dinámicos desde dentro de la función con `get_current_run_tree()`, al invocar con `langsmith_extra` y para todo un bloque con `tracing_context(tags=..., metadata=...)`. Ejecutar varias preguntas y filtrar después las trazas por etiqueta y por metadato, como se haría en la interfaz de LangSmith.

In [3]:
@traceable(run_type="chain", tags=["asistente"], metadata={"version": "1.0"})
def responder_con_info(pregunta: str) -> str:
    respuesta = responder.__wrapped__(pregunta)
    traza_actual = ls.get_current_run_tree()
    traza_actual.metadata["longitud"] = len(respuesta)
    if "capital" in pregunta.lower():
        traza_actual.tags.append("capitales")
    return respuesta


trazas = []
with ls.tracing_context(enabled="local", tags=["pruebas"], metadata={"entorno": "notebook"}):
    for i, pregunta in enumerate(["¿Cuánto mide el Teide?", "¿Capital de Lanzarote?", "¿Dónde está el Roque Nublo?", "¿Capital de La Gomera?"]):
        responder_con_info(pregunta, langsmith_extra={"on_end": trazas.append, "metadata": {"usuario": f"u{i % 2}"}})

for t in trazas:
    print(f"{t.inputs['pregunta']:<30} tags={sorted(t.tags)} metadata={ {k: v for k, v in t.metadata.items() if k != 'ls_method'} }")
capitales = [t.inputs["pregunta"] for t in trazas if "capitales" in t.tags]
de_u1 = [t.inputs["pregunta"] for t in trazas if t.metadata["usuario"] == "u1"]
print("Con la etiqueta 'capitales':", capitales)
print("Del usuario u1:", de_u1)
assert len(capitales) == 2 and all("pruebas" in t.tags for t in trazas)

¿Cuánto mide el Teide?         tags=['asistente', 'pruebas'] metadata={'usuario': 'u0', 'entorno': 'notebook', 'version': '1.0', 'longitud': 26}
¿Capital de Lanzarote?         tags=['asistente', 'capitales', 'pruebas'] metadata={'usuario': 'u1', 'entorno': 'notebook', 'version': '1.0', 'longitud': 36}
¿Dónde está el Roque Nublo?    tags=['asistente', 'pruebas'] metadata={'usuario': 'u0', 'entorno': 'notebook', 'version': '1.0', 'longitud': 36}
¿Capital de La Gomera?         tags=['asistente', 'capitales', 'pruebas'] metadata={'usuario': 'u1', 'entorno': 'notebook', 'version': '1.0', 'longitud': 54}
Con la etiqueta 'capitales': ['¿Capital de Lanzarote?', '¿Capital de La Gomera?']
Del usuario u1: ['¿Capital de Lanzarote?', '¿Capital de La Gomera?']


## 4. Trazar un bloque de código

No todo es una función: con el gestor de contexto `ls.trace` se traza un bloque, se le pueden añadir hijos (funciones `@traceable` llamadas dentro) y se cierra con `end(outputs=...)`. Trazar un lote de preguntas como un solo paso y comprobar que cada pregunta aparece como hijo.

In [4]:
preguntas = ["¿Capital de Fuerteventura?", "¿Cuánto mide el Teide?", "¿Cuál es la isla habitada más pequeña?"]
with ls.tracing_context(enabled="local"):
    with ls.trace("Lote de preguntas", run_type="chain", inputs={"preguntas": preguntas}) as lote:
        respuestas = [responder(p) for p in preguntas]
        lote.end(outputs={"respuestas": respuestas})

mostrar_arbol(lote)
print(lote.outputs)
assert len(lote.child_runs) == len(preguntas)

Lote de preguntas [chain] 32.3 ms
    responder [chain] 10.8 ms
        normalizar [chain] 0.1 ms
        recuperar [retriever] 0.1 ms
        generar [llm] 10.5 ms
    responder [chain] 10.7 ms
        normalizar [chain] 0.1 ms
        recuperar [retriever] 0.1 ms
        generar [llm] 10.2 ms
    responder [chain] 10.6 ms
        normalizar [chain] 0.1 ms
        recuperar [retriever] 0.1 ms
        generar [llm] 10.2 ms
{'respuestas': ['La capital de Fuerteventura es Puerto del Rosario.', 'El Teide mide 3715 metros.', 'La Graciosa es la isla habitada más pequeña.']}


## 5. Ocultar datos sensibles

Las trazas guardan entradas y salidas tal cual, así que pueden contener datos personales. Usar `process_inputs` y `process_outputs` en `@traceable` para sustituir correos y teléfonos por marcadores antes de registrar la traza, y comprobar que la función sigue recibiendo los datos reales.

In [5]:
CORREO = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
TELEFONO = re.compile(r"\b\d{3}[ -]?\d{3}[ -]?\d{3}\b")


def anonimizar(texto):
    return TELEFONO.sub("<TELÉFONO>", CORREO.sub("<CORREO>", texto))


@traceable(
    process_inputs=lambda entradas: {k: anonimizar(v) for k, v in entradas.items()},
    process_outputs=lambda salida: {"output": anonimizar(salida)},
)
def registrar_reserva(cliente: str, contacto: str) -> str:
    assert "@" in contacto or contacto[0].isdigit()
    return f"Reserva para {cliente}; confirmación enviada a {contacto}"


trazas = []
with ls.tracing_context(enabled="local"):
    for contacto in ["yaiza.perez@example.com", "928 123 456"]:
        print(registrar_reserva("Yaiza", contacto, langsmith_extra={"on_end": trazas.append}))
for t in trazas:
    print("traza:", t.inputs, "->", t.outputs)
assert all("@" not in str(t.inputs) + str(t.outputs) and "928" not in str(t.inputs) for t in trazas)

Reserva para Yaiza; confirmación enviada a yaiza.perez@example.com
Reserva para Yaiza; confirmación enviada a 928 123 456
traza: {'cliente': 'Yaiza', 'contacto': '<CORREO>'} -> {'output': 'Reserva para Yaiza; confirmación enviada a <CORREO>'}
traza: {'cliente': 'Yaiza', 'contacto': '<TELÉFONO>'} -> {'output': 'Reserva para Yaiza; confirmación enviada a <TELÉFONO>'}


## 6. Errores en las trazas

Cuando una función trazada lanza una excepción, la traza se cierra igualmente y guarda el error, tanto en el paso que falla como en sus padres. Provocar un error en el paso `generar` (sin documentos), capturar la excepción e imprimir la última línea del error registrado en cada nivel del árbol.

In [6]:
@traceable(run_type="chain")
def responder_sin_documentos(pregunta: str) -> str:
    return generar(pregunta, recuperar(normalizar(pregunta), k=0))


trazas = []
with ls.tracing_context(enabled="local"):
    try:
        responder_sin_documentos("¿Capital de Marte?", langsmith_extra={"on_end": trazas.append})
    except IndexError as e:
        print("Excepción capturada:", repr(e))


def errores(traza, nivel=0):
    print(f"{'    ' * nivel}{traza.name}: {traza.error.strip().splitlines()[-1] if traza.error else 'sin error'}")
    for hijo in traza.child_runs:
        errores(hijo, nivel + 1)


errores(trazas[0])
assert trazas[0].error and trazas[0].child_runs[-1].error

Excepción capturada: IndexError('list index out of range')
responder_sin_documentos: IndexError: list index out of range
    normalizar: sin error
    recuperar: sin error
    generar: IndexError: list index out of range


## 7. Trazas de LangChain

Los componentes de LangChain generan sus propias trazas mediante *callbacks*. Ejecutar una cadena `prompt | modelo | parser` (con un modelo de prueba) dentro de `collect_runs()` de `langchain_core` y recorrer el árbol de ejecuciones que recibiría LangSmith: tipo de cada paso, prompt que llegó al modelo, texto generado y salida final.

In [7]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tracers.context import collect_runs

cadena = (
    ChatPromptTemplate.from_messages([("system", "Eres un guía de Canarias."), ("human", "{pregunta}")])
    | GenericFakeChatModel(messages=iter([AIMessage("Valverde es la capital de El Hierro.")]))
    | StrOutputParser()
)
with collect_runs() as recolector:
    cadena.invoke({"pregunta": "¿Capital de El Hierro?"}, config={"run_name": "Guía", "tags": ["langchain"]})

raiz = recolector.traced_runs[0]
mostrar_arbol(raiz)
modelo = next(r for r in raiz.child_runs if r.run_type == "llm")
print("Prompt que recibió el modelo:", modelo.inputs["prompts"])
print("Texto generado:", modelo.outputs["generations"][0][0]["text"])
print("Salida:", raiz.outputs, "| etiquetas:", raiz.tags)

Guía [chain] 2.4 ms
    ChatPromptTemplate [prompt] 0.2 ms
    GenericFakeChatModel [llm] 1.5 ms
    StrOutputParser [parser] 0.1 ms
Prompt que recibió el modelo: ['System: Eres un guía de Canarias.\nHuman: ¿Capital de El Hierro?']
Texto generado: Valverde es la capital de El Hierro.
Salida: {'output': 'Valverde es la capital de El Hierro.'} | etiquetas: ['langchain']


## 8. Conjunto de datos

Guardar en `asistente.py` (con `%%writefile`) dos versiones de un asistente de preguntas sobre Canarias: `responder_v1` busca el hecho que comparte más palabras con la pregunta y `responder_v2` además normaliza tildes, usa sinónimos y responde "No lo sé" cuando no encuentra nada. Crear un conjunto de 10 ejemplos de LangSmith (`Example`) con la pregunta como entrada, la respuesta esperada como salida de referencia y la dificultad en los metadatos.

In [8]:
%%writefile asistente.py
import re
import unicodedata

HECHOS = [
    "La capital de Lanzarote es Arrecife.",
    "La capital de Fuerteventura es Puerto del Rosario.",
    "La capital de La Gomera es San Sebastián de La Gomera.",
    "La capital de El Hierro es Valverde.",
    "El Teide mide 3715 metros.",
    "El Roque Nublo está en Gran Canaria.",
    "La Graciosa es la isla habitada más pequeña.",
]
SINONIMOS = {"altura": "mide", "alto": "mide", "chica": "pequeña", "pequena": "pequeña", "ubicado": "está", "donde": "está"}


def palabras(texto):
    return {p for p in re.findall(r"\w+", texto.lower()) if len(p) > 3}


def sin_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")


def responder_v1(inputs: dict) -> dict:
    clave = palabras(inputs["pregunta"])
    return {"respuesta": max(HECHOS, key=lambda h: len(clave & palabras(h)))}


def responder_v2(inputs: dict) -> dict:
    clave = {sin_tildes(SINONIMOS.get(sin_tildes(p), p)) for p in palabras(inputs["pregunta"])}
    puntuados = [(len(clave & {sin_tildes(p) for p in palabras(h)}), h) for h in HECHOS]
    puntos, hecho = max(puntuados, key=lambda x: x[0])
    return {"respuesta": hecho if puntos else "No lo sé."}

Overwriting asistente.py

In [9]:
import uuid

from asistente import responder_v1, responder_v2
from langsmith.schemas import Example

DATOS = [
    ("¿Cuál es la capital de Lanzarote?", "Arrecife", "facil"),
    ("¿Cuál es la capital de El Hierro?", "Valverde", "facil"),
    ("¿Cuánto mide el Teide?", "3715 metros", "facil"),
    ("¿Capital de Fuerteventura?", "Puerto del Rosario", "facil"),
    ("¿En qué isla está el Roque Nublo?", "Gran Canaria", "media"),
    ("¿Cuál es la isla habitada más pequeña?", "La Graciosa", "media"),
    ("¿Qué altura tiene el pico más alto de Canarias?", "3715 metros", "dificil"),
    ("¿Cual es la isla habitada mas chica?", "La Graciosa", "dificil"),
    ("¿Dónde queda el Roque Nublo?", "Gran Canaria", "dificil"),
    ("¿Quién ganó la liga en 1990?", "No lo sé", "dificil"),
]
id_conjunto = uuid.uuid5(uuid.NAMESPACE_DNS, "canarias-qa")
ejemplos = [
    Example(id=uuid.uuid5(id_conjunto, p), dataset_id=id_conjunto, inputs={"pregunta": p}, outputs={"respuesta": r}, metadata={"dificultad": d})
    for p, r, d in DATOS
]
print(len(ejemplos), "ejemplos")
print(ejemplos[0].inputs, ejemplos[0].outputs, ejemplos[0].metadata)
print(responder_v1(ejemplos[6].inputs), responder_v2(ejemplos[6].inputs))

10 ejemplos
{'pregunta': '¿Cuál es la capital de Lanzarote?'} {'respuesta': 'Arrecife'} {'dificultad': 'facil'}
{'respuesta': 'La capital de Lanzarote es Arrecife.'} {'respuesta': 'El Teide mide 3715 metros.'}


## 9. Evaluadores

Escribir tres evaluadores con la firma que espera LangSmith (`outputs`, `reference_outputs` y opcionalmente `inputs`):
- `contiene`: la respuesta contiene la referencia sin tener en cuenta mayúsculas ni tildes (booleano).
- `concisa`: la respuesta tiene como mucho 12 palabras (devuelve un diccionario con `score` y `comment`).
- `juez`: un "LLM como juez" que puntúa de 0 a 1 la respuesta frente a la referencia. El juez es un modelo de chat local con reglas que responde en JSON; su llamada se hace con el trazado desactivado para que no intente enviarse a LangSmith sin clave.

Probarlos a mano con un acierto y un fallo.

In [10]:
import json

from langchain_core.language_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult


def normal(texto):
    return sin_tildes(texto.lower()).rstrip(".")


def contiene(outputs: dict, reference_outputs: dict) -> bool:
    return normal(reference_outputs["respuesta"]) in normal(outputs["respuesta"])


def concisa(outputs: dict) -> dict:
    n = len(outputs["respuesta"].split())
    return {"key": "concisa", "score": n <= 12, "comment": f"{n} palabras"}


class ModeloJuez(BaseChatModel):
    @property
    def _llm_type(self) -> str:
        return "juez"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        m = re.search(r"Referencia: (.*)\nRespuesta: (.*)", messages[-1].text)
        referencia, respuesta = (set(re.findall(r"\w+", normal(t))) for t in m.groups())
        cubierto = len(referencia & respuesta) / len(referencia)
        veredicto = {"puntuacion": round(cubierto, 2), "razon": "coincide con la referencia" if cubierto == 1 else f"cubre el {cubierto:.0%} de la referencia"}
        return ChatResult(generations=[ChatGeneration(message=AIMessage(json.dumps(veredicto, ensure_ascii=False)))])


prompt_juez = ChatPromptTemplate.from_template(
    "Puntúa de 0 a 1 si la respuesta contiene la información de la referencia. Responde en JSON.\nReferencia: {referencia}\nRespuesta: {respuesta}"
)
cadena_juez = prompt_juez | ModeloJuez() | StrOutputParser() | json.loads


def juez(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    with ls.tracing_context(enabled=False):
        veredicto = cadena_juez.invoke({"referencia": reference_outputs["respuesta"], "respuesta": outputs["respuesta"]})
    return {"key": "juez", "score": veredicto["puntuacion"], "comment": veredicto["razon"]}


for ejemplo in (ejemplos[0], ejemplos[6]):
    salida = responder_v1(ejemplo.inputs)
    print(ejemplo.inputs["pregunta"], "->", salida["respuesta"])
    print("   ", contiene(salida, ejemplo.outputs), concisa(salida), juez(ejemplo.inputs, salida, ejemplo.outputs))

¿Cuál es la capital de Lanzarote? -> La capital de Lanzarote es Arrecife.
    True {'key': 'concisa', 'score': True, 'comment': '6 palabras'} {'key': 'juez', 'score': 1.0, 'comment': 'coincide con la referencia'}
¿Qué altura tiene el pico más alto de Canarias? -> La capital de Lanzarote es Arrecife.
    False {'key': 'concisa', 'score': True, 'comment': '6 palabras'} {'key': 'juez', 'score': 0.0, 'comment': 'cubre el 0% de la referencia'}


## 10. Evaluación local

Evaluar `responder_v1` sobre el conjunto con `evaluate(..., upload_results=False)` y los tres evaluadores, más un evaluador de resumen (`summary_evaluators`) que calcule el porcentaje de aciertos de todo el experimento. Convertir el resultado en un `DataFrame` con `to_pandas()`, mostrar las columnas principales y la media de cada evaluador por dificultad.

In [11]:
import contextlib
import io

from langsmith import evaluate


def tasa_aciertos(outputs: list[dict], reference_outputs: list[dict]) -> dict:
    aciertos = sum(contiene(o, r) for o, r in zip(outputs, reference_outputs))
    return {"key": "tasa_aciertos", "score": aciertos / len(outputs)}


def experimento(objetivo, prefijo):
    # evaluate imprime el nombre aleatorio del experimento; se silencia para que la salida sea reproducible
    with contextlib.redirect_stdout(io.StringIO()):
        return evaluate(
            objetivo, data=ejemplos, evaluators=[contiene, concisa, juez], summary_evaluators=[tasa_aciertos],
            experiment_prefix=prefijo, upload_results=False, max_concurrency=0,
        )


resultados_v1 = experimento(responder_v1, "v1")
tabla_v1 = resultados_v1.to_pandas()
tabla_v1["dificultad"] = [e.metadata["dificultad"] for e in ejemplos]
columnas = ["inputs.pregunta", "outputs.respuesta", "feedback.contiene", "feedback.concisa", "feedback.juez"]
print(tabla_v1[columnas].to_string(max_colwidth=40))
print(tabla_v1.groupby("dificultad")[["feedback.contiene", "feedback.juez"]].mean())
resumen = resultados_v1._summary_results["results"][0]
print(f"{resumen.key}: {resumen.score:.0%}")

                           inputs.pregunta                        outputs.respuesta  feedback.contiene  feedback.concisa  feedback.juez
0        ¿Cuál es la capital de Lanzarote?     La capital de Lanzarote es Arrecife.               True              True            1.0
1        ¿Cuál es la capital de El Hierro?     La capital de El Hierro es Valverde.               True              True            1.0
2                   ¿Cuánto mide el Teide?               El Teide mide 3715 metros.               True              True            1.0
3               ¿Capital de Fuerteventura?  La capital de Fuerteventura es Puert...               True              True            1.0
4        ¿En qué isla está el Roque Nublo?     El Roque Nublo está en Gran Canaria.               True              True            1.0
5   ¿Cuál es la isla habitada más pequeña?  La Graciosa es la isla habitada más ...               True              True            1.0
6  ¿Qué altura tiene el pico más alto d...     L

## 11. Comparar experimentos

Evaluar también `responder_v2` y comparar los dos experimentos: media de cada evaluador en una tabla y lista de los ejemplos que mejoran o empeoran según `contiene`. Comprobar que la versión 2 no empeora ningún ejemplo.

In [12]:
import pandas as pd

tabla_v2 = experimento(responder_v2, "v2").to_pandas()
metricas = ["feedback.contiene", "feedback.concisa", "feedback.juez"]
comparacion = pd.DataFrame({"v1": tabla_v1[metricas].astype(float).mean(), "v2": tabla_v2[metricas].astype(float).mean()})
comparacion["diferencia"] = comparacion["v2"] - comparacion["v1"]
print(comparacion.round(2))

cambios = tabla_v2["feedback.contiene"].astype(int) - tabla_v1["feedback.contiene"].astype(int)
for i in cambios[cambios != 0].index:
    print(f"{'mejora ' if cambios[i] > 0 else 'empeora'} {tabla_v1['inputs.pregunta'][i]:<40} v1: {tabla_v1['outputs.respuesta'][i]} | v2: {tabla_v2['outputs.respuesta'][i]}")
assert (cambios >= 0).all() and comparacion.loc["feedback.contiene", "diferencia"] > 0

                    v1   v2  diferencia
feedback.contiene  0.8  1.0         0.2
feedback.concisa   1.0  1.0         0.0
feedback.juez      0.8  1.0         0.2
mejora  ¿Qué altura tiene el pico más alto de Canarias? v1: La capital de Lanzarote es Arrecife. | v2: El Teide mide 3715 metros.
mejora  ¿Quién ganó la liga en 1990?             v1: La capital de Lanzarote es Arrecife. | v2: No lo sé.


## 12. Pruebas con pytest

LangSmith se integra con `pytest`: los tests marcados con `@pytest.mark.langsmith` registran entradas, salidas, referencias y *feedback* con `langsmith.testing`, y con una clave se guardan como un experimento. Escribir `test_asistente.py` con un test parametrizado sobre los ejemplos fáciles y medios que use `responder_v2`, y ejecutarlo con `LANGSMITH_TEST_TRACKING=false` para no enviar nada.

In [13]:
%%writefile test_asistente.py
import pytest
from langsmith import testing as t

from asistente import responder_v2

CASOS = [
    ("¿Cuál es la capital de Lanzarote?", "Arrecife"),
    ("¿Cuánto mide el Teide?", "3715 metros"),
    ("¿En qué isla está el Roque Nublo?", "Gran Canaria"),
    ("¿Qué altura tiene el Teide?", "3715 metros"),
]


@pytest.mark.langsmith
@pytest.mark.parametrize("pregunta, esperada", CASOS)
def test_responde_con_el_dato(pregunta, esperada):
    t.log_inputs({"pregunta": pregunta})
    t.log_reference_outputs({"respuesta": esperada})
    respuesta = responder_v2({"pregunta": pregunta})["respuesta"]
    t.log_outputs({"respuesta": respuesta})
    acierto = esperada.lower() in respuesta.lower()
    t.log_feedback(key="contiene", score=acierto)
    assert acierto

Overwriting test_asistente.py


In [14]:
import os
import subprocess
import sys

proceso = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", "test_asistente.py"],
    capture_output=True, text=True, env={**os.environ, "LANGSMITH_TEST_TRACKING": "false"},
)
print(re.sub(r" in [\d.]+s", "", proceso.stdout.strip()))
assert proceso.returncode == 0

....                                                                     [100%]
4 passed
